# Week 18: Final Project Development & Presentation
# 第 18 週：綜合專題開發與展示

## 學習目標 Learning Objectives
1. 建立完整的端到端 (End-to-End) ML Pipeline
2. 實作自動化探索式資料分析 (EDA)
3. 使用 Pipeline 與 ColumnTransformer 進行前處理
4. 訓練多個模型並建立比較框架
5. 執行超參數調校與驗證曲線
6. 綜合評估模型：混淆矩陣、ROC、特徵重要性
7. 建立結果視覺化儀表板
8. 實作模型解釋與部分依賴圖
9. 產生結構化報告
10. 18 週課程總回顧

**所需套件 Required packages:** numpy, pandas, matplotlib, scikit-learn, scipy

---

In [ ]:
# === Imports and Setup ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import label_binarize
from scipy import stats
from itertools import cycle

# Plot settings
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('All packages imported successfully!')
print('\u6240\u6709\u5957\u4ef6\u532f\u5165\u6210\u529f\uff01')

---
## Part 1: End-to-End ML Pipeline Template
## 第一部分：端到端 ML Pipeline 模板

一個完整的 ML 專案流程包括：

1. **問題定義** Problem Definition
2. **資料載入** Data Loading
3. **EDA** Exploratory Data Analysis
4. **前處理** Preprocessing
5. **模型訓練** Model Training
6. **評估** Evaluation
7. **調參** Hyperparameter Tuning
8. **最終報告** Final Report

In [ ]:
# === Step 1: Data Loading ===
# Using the Wine Quality dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')
target_names = wine.target_names

# Combine for EDA
df = X.copy()
df['target'] = y
df['target_name'] = [target_names[t] for t in y]

print('=== Dataset Overview ===')
print(f'Shape: {X.shape}')
print(f'Features: {X.shape[1]}')
print(f'Samples: {X.shape[0]}')
print(f'Classes: {list(target_names)}')
print(f'\nClass Distribution:')
print(y.value_counts().sort_index())
print(f'\nFirst 5 rows:')
print(X.head())
print(f'\nBasic Statistics:')
print(X.describe().round(2))

---
## Part 2: Automated EDA Template
## 第二部分：自動化 EDA 模板

探索式資料分析 (EDA) 是任何 ML 專案的第一步。包括分布圖、相關矩陣、缺失值分析。

EDA is the first step of any ML project: distribution plots, correlation matrix, missing value analysis.

In [ ]:
# === Automated EDA ===
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (1) Class distribution
ax = axes[0, 0]
class_counts = y.value_counts().sort_index()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(target_names, class_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, str(val), ha='center', fontweight='bold')
ax.set_title('Class Distribution / \u985e\u5225\u5206\u5e03')
ax.set_ylabel('Count')

# (2) Feature distributions (top 6 features by variance)
ax = axes[0, 1]
top_features = X.var().nlargest(6).index
for i, feat in enumerate(top_features[:3]):
    ax.hist(X[feat], bins=20, alpha=0.5, label=feat[:15], density=True)
ax.set_title('Top Feature Distributions / \u4e3b\u8981\u7279\u5fb5\u5206\u5e02')
ax.legend(fontsize=8)
ax.set_ylabel('Density')

# (3) Correlation matrix
ax = axes[0, 2]
corr = X.corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(X.columns)))
ax.set_xticklabels([c[:6] for c in X.columns], rotation=90, fontsize=6)
ax.set_yticks(range(len(X.columns)))
ax.set_yticklabels([c[:6] for c in X.columns], fontsize=6)
ax.set_title('Correlation Matrix / \u76f8\u95dc\u77e9\u9663')
plt.colorbar(im, ax=ax, shrink=0.8)

# (4) Missing values
ax = axes[1, 0]
missing = X.isnull().sum()
ax.barh(range(len(missing)), missing.values, color='#FFA07A', edgecolor='white')
ax.set_yticks(range(len(missing)))
ax.set_yticklabels([c[:12] for c in X.columns], fontsize=7)
ax.set_title('Missing Values / \u7f3a\u5931\u503c')
ax.set_xlabel('Count')
if missing.sum() == 0:
    ax.text(0.5, 0.5, 'No missing values!', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='green', fontweight='bold')

# (5) Box plots for top features by class
ax = axes[1, 1]
feat_to_plot = X.columns[0]  # alcohol
data_by_class = [X[feat_to_plot][y == c].values for c in range(3)]
bp = ax.boxplot(data_by_class, labels=target_names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title(f'Box Plot: {feat_to_plot} by Class / \u7bb1\u5f62\u5716')
ax.set_ylabel(feat_to_plot)

# (6) Feature pair scatter
ax = axes[1, 2]
f1_name, f2_name = X.columns[0], X.columns[6]  # alcohol, flavanoids
for c, color, name in zip(range(3), colors, target_names):
    mask = y == c
    ax.scatter(X[f1_name][mask], X[f2_name][mask], c=color, label=name, alpha=0.6, s=30, edgecolors='white')
ax.set_xlabel(f1_name)
ax.set_ylabel(f2_name)
ax.set_title(f'{f1_name} vs {f2_name} / \u7279\u5fb5\u6563\u5e03\u5716')
ax.legend(fontsize=9)

plt.suptitle('Automated EDA Dashboard / \u81ea\u52d5\u5316\u63a2\u7d22\u5f0f\u5206\u6790\u5100\u8868\u677f', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics
print('=== EDA Summary ===')
print(f'Total features: {X.shape[1]}')
print(f'Total samples: {X.shape[0]}')
print(f'Missing values: {X.isnull().sum().sum()}')
print(f'\nHighly correlated pairs (|r| > 0.8):')
high_corr = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))
for f1, f2, r in high_corr:
    print(f'  {f1} <-> {f2}: r = {r:.3f}')

---
## Part 3: Preprocessing Pipeline
## 第三部分：前處理 Pipeline

使用 `Pipeline` 將前處理與模型訓練封裝在一起，避免資料洩漏 (Data Leakage)。

Use `Pipeline` to encapsulate preprocessing and model training together, preventing data leakage.

$$
\text{Pipeline}: X_{\text{raw}} \xrightarrow{\text{Scale}} X_{\text{scaled}} \xrightarrow{\text{Model}} \hat{y}
$$

In [ ]:
# === Preprocessing Pipeline ===
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

# All features are numeric in the Wine dataset
numeric_features = X.columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
    ],
    remainder='passthrough'
)

# Quick test with LogisticRegression
test_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])
test_pipe.fit(X_train, y_train)
print(f'\nPipeline test - LogisticRegression accuracy: {test_pipe.score(X_test, y_test):.4f}')
print('Pipeline setup validated successfully!')

---
## Part 4: Model Comparison Framework
## 第四部分：模型比較框架

訓練 5 個以上的模型，建立比較表與圖表。

Train 5+ models and create a comprehensive comparison table and chart.

In [ ]:
# === Model Comparison Framework ===
model_configs = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

comparison_results = []
trained_models = {}

for name, model in model_configs.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Fit and evaluate
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    train_acc = pipe.score(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Cross-validation
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    
    comparison_results.append({
        'Model': name,
        'Train Acc': train_acc,
        'Test Acc': test_acc,
        'F1 (weighted)': f1,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Overfit Gap': train_acc - test_acc,
    })
    trained_models[name] = pipe

df_comparison = pd.DataFrame(comparison_results).sort_values('CV Mean', ascending=False)
print('=== Model Comparison Results ===')
print(df_comparison.to_string(index=False, float_format='%.4f'))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) Test Accuracy comparison
sorted_df = df_comparison.sort_values('Test Acc')
colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_df)))
axes[0].barh(sorted_df['Model'], sorted_df['Test Acc'], color=colors_bar, edgecolor='white', height=0.6)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Test Accuracy / \u6e2c\u8a66\u6e96\u78ba\u7387')
axes[0].set_xlim(0.8, 1.02)
for i, (_, row) in enumerate(sorted_df.iterrows()):
    axes[0].text(row['Test Acc'] + 0.003, i, f'{row["Test Acc"]:.3f}', va='center', fontsize=8)

# (2) CV scores with error bars
axes[1].barh(sorted_df['Model'], sorted_df['CV Mean'], xerr=sorted_df['CV Std'],
             color=colors_bar, edgecolor='white', height=0.6, capsize=3)
axes[1].set_xlabel('CV Accuracy (mean +/- std)')
axes[1].set_title('Cross-Validation / \u4ea4\u53c9\u9a57\u8b49')
axes[1].set_xlim(0.8, 1.02)

# (3) Overfitting gap
overfit_colors = ['#FF6B6B' if g > 0.05 else '#4CAF50' for g in sorted_df['Overfit Gap']]
axes[2].barh(sorted_df['Model'], sorted_df['Overfit Gap'], color=overfit_colors, edgecolor='white', height=0.6)
axes[2].axvline(x=0.05, color='red', linestyle='--', alpha=0.5, label='Warning threshold')
axes[2].set_xlabel('Train-Test Gap')
axes[2].set_title('Overfitting Gap / \u904e\u64ec\u5408\u5dee\u8ddd')
axes[2].legend(fontsize=8)

plt.suptitle('Model Comparison Framework / \u6a21\u578b\u6bd4\u8f03\u6846\u67b6', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

best_model_name = df_comparison.iloc[0]['Model']
print(f'\nBest model by CV: {best_model_name} (CV Mean = {df_comparison.iloc[0]["CV Mean"]:.4f})')

---
## Part 5: Hyperparameter Tuning
## 第五部分：超參數調校

使用 GridSearchCV 對最佳模型進行超參數搜尋，並繪製驗證曲線。

Use GridSearchCV on the best model and plot validation curves.

In [ ]:
# === Hyperparameter Tuning with GridSearchCV ===
# Tune RandomForest (commonly the best or near-best)
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 20, None],
    'classifier__min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    rf_pipe, param_grid, cv=5, scoring='accuracy',
    n_jobs=-1, return_train_score=True, verbose=0
)
grid_search.fit(X_train, y_train)

print('=== GridSearchCV Results ===')
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_:.4f}')
print(f'Test Score: {grid_search.score(X_test, y_test):.4f}')

# Top 5 parameter combinations
cv_results = pd.DataFrame(grid_search.cv_results_)
top5 = cv_results.nsmallest(5, 'rank_test_score')[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']]
print('\nTop 5 Parameter Combinations:')
for _, row in top5.iterrows():
    print(f'  {row["params"]} -> CV: {row["mean_test_score"]:.4f} (+/- {row["std_test_score"]:.4f})')

# Validation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Validation curve for n_estimators
n_est_range = [10, 25, 50, 75, 100, 150, 200, 300]
train_scores, test_scores = validation_curve(
    Pipeline([('preprocessor', preprocessor),
              ('classifier', RandomForestClassifier(random_state=42))]),
    X, y, param_name='classifier__n_estimators',
    param_range=n_est_range, cv=5, scoring='accuracy'
)

axes[0].plot(n_est_range, train_scores.mean(axis=1), 'o-', color='#FF6B6B', label='Train')
axes[0].fill_between(n_est_range, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='#FF6B6B')
axes[0].plot(n_est_range, test_scores.mean(axis=1), 'o-', color='#4ECDC4', label='Validation')
axes[0].fill_between(n_est_range, test_scores.mean(axis=1) - test_scores.std(axis=1),
                     test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.1, color='#4ECDC4')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Validation Curve: n_estimators\n\u9a57\u8b49\u66f2\u7dda')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (2) Learning curve for best model
best_pipe = grid_search.best_estimator_
train_sizes, train_lc, test_lc = learning_curve(
    best_pipe, X, y, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

axes[1].plot(train_sizes, train_lc.mean(axis=1), 'o-', color='#FF6B6B', label='Train')
axes[1].fill_between(train_sizes, train_lc.mean(axis=1) - train_lc.std(axis=1),
                     train_lc.mean(axis=1) + train_lc.std(axis=1), alpha=0.1, color='#FF6B6B')
axes[1].plot(train_sizes, test_lc.mean(axis=1), 'o-', color='#4ECDC4', label='Validation')
axes[1].fill_between(train_sizes, test_lc.mean(axis=1) - test_lc.std(axis=1),
                     test_lc.mean(axis=1) + test_lc.std(axis=1), alpha=0.1, color='#4ECDC4')
axes[1].set_xlabel('Training Samples')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Learning Curve (Best Model)\n\u5b78\u7fd2\u66f2\u7dda')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Tuning / \u8d85\u53c3\u6578\u8abf\u6821', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 6: Final Model Evaluation
## 第六部分：最終模型評估

全面評估最佳模型：混淆矩陣、分類報告、ROC 曲線、特徵重要性。

Comprehensive evaluation: confusion matrix, classification report, ROC curves, feature importance.

In [ ]:
# === Final Model Evaluation ===
final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test)

# Get the classifier from the pipeline for feature importance
rf_classifier = final_model.named_steps['classifier']

print('=== Classification Report ===')
print(classification_report(y_test, y_pred_final, target_names=target_names))

# Multi-class ROC
# Binarize the output
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = 3

# Get probability predictions
y_score = final_model.predict_proba(X_test)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# (1) Confusion Matrix
ax = axes[0, 0]
cm = confusion_matrix(y_test, y_pred_final)
im = ax.imshow(cm, cmap='Blues')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=18, color=color, fontweight='bold')
ax.set_xticks(range(3))
ax.set_xticklabels(target_names)
ax.set_yticks(range(3))
ax.set_yticklabels(target_names)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix / \u6df7\u6dc6\u77e9\u9663', fontsize=13)

# (2) ROC Curves (one-vs-rest)
ax = axes[0, 1]
colors_roc = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for i, (color, name) in enumerate(zip(colors_roc, target_names)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (One-vs-Rest) / ROC \u66f2\u7dda', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (3) Feature Importance
ax = axes[1, 0]
importances = rf_classifier.feature_importances_
sorted_idx = np.argsort(importances)
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color='#4ECDC4', edgecolor='white')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([wine.feature_names[i][:15] for i in sorted_idx], fontsize=8)
ax.set_xlabel('Importance')
ax.set_title('Feature Importance / \u7279\u5fb5\u91cd\u8981\u6027', fontsize=13)

# (4) Prediction confidence distribution
ax = axes[1, 1]
max_probs = y_score.max(axis=1)
correct = (y_pred_final == y_test)
ax.hist(max_probs[correct], bins=20, alpha=0.6, color='#4CAF50', label='Correct', density=True)
ax.hist(max_probs[~correct], bins=20, alpha=0.6, color='#F44336', label='Incorrect', density=True)
ax.set_xlabel('Prediction Confidence')
ax.set_ylabel('Density')
ax.set_title('Confidence Distribution / \u4fe1\u5fc3\u5ea6\u5206\u5e03', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Final Model Evaluation / \u6700\u7d42\u6a21\u578b\u8a55\u4f30', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 7: Results Visualization Dashboard
## 第七部分：結果視覺化儀表板

建立一個 2x3 的綜合圖表，摘要所有關鍵結果（類似海報\u5c55\u793a\uff09。

Create a 2x3 subplot figure summarizing all key results (like a poster).

In [ ]:
# === Results Dashboard (Poster-style) ===
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# (1) Dataset overview - class distribution
ax = axes[0, 0]
wedges, texts, autotexts = ax.pie(
    class_counts.values, labels=target_names, colors=['#FF6B6B', '#4ECDC4', '#45B7D1'],
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax.set_title('Dataset Distribution\n\u8cc7\u6599\u96c6\u5206\u5e02', fontsize=13, fontweight='bold')

# (2) Model comparison summary
ax = axes[0, 1]
top_models = df_comparison.head(5)
y_pos = range(len(top_models))
bars = ax.barh(y_pos, top_models['CV Mean'], xerr=top_models['CV Std'],
               color=plt.cm.Set2(np.linspace(0, 1, 5)), edgecolor='white', height=0.6, capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_models['Model'].values, fontsize=9)
ax.set_xlabel('CV Accuracy')
ax.set_title('Top 5 Models (CV Score)\n\u524d 5 \u540d\u6a21\u578b', fontsize=13, fontweight='bold')
ax.set_xlim(0.9, 1.01)

# (3) Best model confusion matrix
ax = axes[0, 2]
im = ax.imshow(cm, cmap='Blues')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, color=color, fontweight='bold')
ax.set_xticks(range(3))
ax.set_xticklabels(target_names, fontsize=9)
ax.set_yticks(range(3))
ax.set_yticklabels(target_names, fontsize=9)
ax.set_title('Confusion Matrix (Best Model)\n\u6df7\u6dc6\u77e9\u9663', fontsize=13, fontweight='bold')

# (4) ROC Curves
ax = axes[1, 0]
for i, (color, name) in enumerate(zip(colors_roc, target_names)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc_val:.2f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curves\nROC \u66f2\u7dda', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (5) Top 10 Feature Importance
ax = axes[1, 1]
top_n = 10
top_idx = sorted_idx[-top_n:]
ax.barh(range(top_n), importances[top_idx],
        color=plt.cm.viridis(np.linspace(0.3, 0.9, top_n)), edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels([wine.feature_names[i] for i in top_idx], fontsize=8)
ax.set_xlabel('Importance')
ax.set_title('Top 10 Features\n\u524d 10 \u91cd\u8981\u7279\u5fb5', fontsize=13, fontweight='bold')

# (6) Key metrics summary
ax = axes[1, 2]
ax.axis('off')
best_acc = grid_search.score(X_test, y_test)
best_cv = grid_search.best_score_
best_f1 = f1_score(y_test, y_pred_final, average='weighted')

metrics_text = f"""
KEY METRICS SUMMARY
{'='*30}

Best Model: {best_model_name}

Test Accuracy:    {best_acc:.4f}
CV Accuracy:      {best_cv:.4f}
F1 (weighted):    {best_f1:.4f}

Dataset:
  Samples:  {X.shape[0]}
  Features: {X.shape[1]}
  Classes:  {n_classes}

Best Params:
"""
for k, v in grid_search.best_params_.items():
    metrics_text += f"  {k.split('__')[1]}: {v}\n"

ax.text(0.1, 0.95, metrics_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Results Dashboard / \u7d50\u679c\u6458\u8981\u5100\u8868\u677f', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 8: Model Interpretation
## 第八部分：模型解釋

特徵重要性 (Feature Importance) 與部分依賴圖 (Partial Dependence Plot)。

Feature importance and partial dependence plots help interpret model predictions.

In [ ]:
# === Model Interpretation ===
# Permutation Importance (model-agnostic)
perm_imp = permutation_importance(final_model, X_test, y_test,
                                   n_repeats=10, random_state=42)

# Sort by importance
perm_sorted_idx = perm_imp.importances_mean.argsort()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) Permutation Importance
ax = axes[0]
top_n = 10
top_perm_idx = perm_sorted_idx[-top_n:]
ax.boxplot(perm_imp.importances[top_perm_idx].T, vert=False,
           labels=[wine.feature_names[i] for i in top_perm_idx])
ax.set_xlabel('Decrease in Accuracy')
ax.set_title('Permutation Importance\n\u7f6e\u63db\u91cd\u8981\u6027', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')

# (2) Partial Dependence (manual implementation for top 2 features)
ax = axes[1]
top_2_features = perm_sorted_idx[-2:]
feat_idx = top_2_features[-1]  # Most important feature
feat_name = wine.feature_names[feat_idx]

# Manual PDP: vary one feature, average predictions
X_test_arr = X_test.values if isinstance(X_test, pd.DataFrame) else X_test
feat_values = np.linspace(
    X_test_arr[:, feat_idx].min(),
    X_test_arr[:, feat_idx].max(),
    50
)

pdp_results = {c: [] for c in range(n_classes)}
for val in feat_values:
    X_temp = X_test.copy()
    X_temp.iloc[:, feat_idx] = val
    probs = final_model.predict_proba(X_temp).mean(axis=0)
    for c in range(n_classes):
        pdp_results[c].append(probs[c])

for c, color, name in zip(range(n_classes), colors_roc, target_names):
    ax.plot(feat_values, pdp_results[c], color=color, lw=2, label=name)

ax.set_xlabel(feat_name)
ax.set_ylabel('Average Predicted Probability')
ax.set_title(f'Partial Dependence: {feat_name}\n\u90e8\u5206\u4f9d\u8cf4\u5716', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Model Interpretation / \u6a21\u578b\u89e3\u91cb', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top features
print('=== Top 5 Most Important Features (Permutation) ===')
for idx in perm_sorted_idx[-5:][::-1]:
    print(f'  {wine.feature_names[idx]}: {perm_imp.importances_mean[idx]:.4f} (+/- {perm_imp.importances_std[idx]:.4f})')

---
## Part 9: Project Report Template
## 第九部分：專案報告模板

自動產生結構化報告，包含所有關鍵指標與發現。

Automatically generate a structured report with all metrics and findings.

In [ ]:
# === Structured Project Report ===
report = f"""
{'='*70}
   ML PROJECT REPORT / \u6a5f\u5668\u5b78\u7fd2\u5c08\u6848\u5831\u544a
{'='*70}

1. PROBLEM DEFINITION / \u554f\u984c\u5b9a\u7fa9
   Task: Multi-class classification of wine varieties
   Goal: Predict wine class (class_0, class_1, class_2) from chemical analysis

2. DATASET / \u8cc7\u6599\u96c6
   Source: UCI Wine Dataset (sklearn built-in)
   Samples: {X.shape[0]}
   Features: {X.shape[1]} (all numeric)
   Classes: {n_classes} ({', '.join(target_names)})
   Missing values: {X.isnull().sum().sum()}
   Train/Test split: {X_train.shape[0]}/{X_test.shape[0]} (80/20)

3. MODELS EVALUATED / \u8a55\u4f30\u7684\u6a21\u578b
"""

for _, row in df_comparison.iterrows():
    report += f"   - {row['Model']:<22} CV: {row['CV Mean']:.4f} (+/- {row['CV Std']:.4f})\n"

report += f"""
4. BEST MODEL / \u6700\u4f73\u6a21\u578b
   Model: RandomForest (tuned via GridSearchCV)
   Best Parameters:
"""
for k, v in grid_search.best_params_.items():
    report += f"     {k.split('__')[1]}: {v}\n"

report += f"""
5. FINAL RESULTS / \u6700\u7d42\u7d50\u679c
   Test Accuracy:    {grid_search.score(X_test, y_test):.4f}
   CV Accuracy:      {grid_search.best_score_:.4f}
   F1 (weighted):    {f1_score(y_test, y_pred_final, average='weighted'):.4f}

6. KEY FEATURES / \u95dc\u9375\u7279\u5fb5
"""
for idx in perm_sorted_idx[-5:][::-1]:
    report += f"   - {wine.feature_names[idx]}: importance = {perm_imp.importances_mean[idx]:.4f}\n"

report += f"""
7. HIGHLY CORRELATED FEATURES / \u9ad8\u76f8\u95dc\u7279\u5fb5
"""
for f1_n, f2_n, r in high_corr[:3]:
    report += f"   - {f1_n} <-> {f2_n}: r = {r:.3f}\n"

report += f"""
8. CONCLUSIONS / \u7d50\u8ad6
   - The RandomForest model achieves strong performance on the Wine dataset
   - Feature importance analysis reveals the top predictors
   - The model shows good generalization with small train-test gap
   - Cross-validation confirms robust performance

9. LIMITATIONS / \u9650\u5236
   - Small dataset ({X.shape[0]} samples) limits generalization conclusions
   - No external validation on out-of-distribution data
   - Feature engineering could further improve results

{'='*70}
End of Report
{'='*70}
"""

print(report)

---
## Part 10: Course Review -- 18 Weeks Summary
## 第十部分：課程總回\u9867 -- 18 \u9031\u6458\u8981

\u56de\u9867\u672c\u8ab2\u7a0b 18 \u9031\u7684\u6240\u6709\u6838\u5fc3\u6982\u5ff5\u3002

Review all 18 weeks of core concepts covered in this course.

In [ ]:
# === 18 Weeks Course Timeline Diagram ===
fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(-0.5, 5.5)
ax.axis('off')

weeks_data = [
    (1, 'ML/DL Overview\n& Python Setup', '#FFB3BA'),
    (2, 'EDA &\nVisualization', '#FFDFBA'),
    (3, 'Supervised Learning\nFramework', '#FFFFBA'),
    (4, 'Linear\nRegression', '#BAFFC9'),
    (5, 'Classification &\nLogistic Reg', '#BAE1FF'),
    (6, 'SVM &\nKernel Methods', '#E8BAFF'),
    (7, 'Tree Models\n& Ensemble', '#FFB3BA'),
    (8, 'Model\nExplainability', '#FFDFBA'),
    (9, 'Feature\nEngineering', '#FFFFBA'),
    (10, 'Hyperparameter\nTuning', '#BAFFC9'),
    (11, 'Neural Network\nBasics', '#BAE1FF'),
    (12, 'CNN &\nComputer Vision', '#E8BAFF'),
    (13, 'RNN &\nTransformer', '#FFB3BA'),
    (14, 'Training\nTechniques', '#FFDFBA'),
    (15, 'Evaluation &\nFairness', '#FFFFBA'),
    (16, 'MLOps\nIntroduction', '#BAFFC9'),
    (17, 'LLM &\nEmbeddings', '#BAE1FF'),
    (18, 'Final Project\n& Review', '#E8BAFF'),
]

# Draw in 2 rows of 9
for i, (week, topic, color) in enumerate(weeks_data):
    row = i // 9
    col = i % 9
    x = col + 0.5
    y = 4.0 - row * 2.5
    
    box = FancyBboxPatch((x - 0.45, y - 0.7), 0.9, 1.4,
                          boxstyle='round,pad=0.08', facecolor=color,
                          edgecolor='#666', linewidth=1)
    ax.add_patch(box)
    ax.text(x, y + 0.35, f'W{week}', ha='center', va='center',
            fontsize=10, fontweight='bold', color='#333')
    ax.text(x, y - 0.15, topic, ha='center', va='center',
            fontsize=6, color='#333')

# Draw phase labels
ax.text(4.5, 5.2, 'ML/DL Visualization Course -- 18 Weeks Overview', ha='center',
        fontsize=16, fontweight='bold')
ax.text(4.5, 4.9, '\u6a5f\u5668\u5b78\u7fd2/\u6df1\u5ea6\u5b78\u7fd2\u8996\u89ba\u5316\u8ab2\u7a0b -- 18 \u9031\u7e3d\u89bd', ha='center',
        fontsize=12, color='#666')

# Phase labels
ax.add_patch(FancyBboxPatch((-0.3, 3.8), 4.8, 0.3, boxstyle='round,pad=0.05',
                             facecolor='#FF6B6B', alpha=0.3, edgecolor='none'))
ax.text(2.1, 3.95, 'Phase 1: ML Fundamentals (W1-10)', ha='center', fontsize=9, fontweight='bold')

ax.add_patch(FancyBboxPatch((4.7, 3.8), 4.8, 0.3, boxstyle='round,pad=0.05',
                             facecolor='#4ECDC4', alpha=0.3, edgecolor='none'))
ax.text(7.1, 3.95, 'Phase 2: DL & Advanced (W11-18)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary table
print('\n=== Course Summary ===')
phases = [
    ('ML Foundations', 'W1-10', 'EDA, Regression, Classification, SVM, Trees, Ensemble, Feature Engineering, Tuning'),
    ('DL Foundations', 'W11-14', 'Neural Networks, CNN, RNN, Transformer, Training Techniques'),
    ('Applications', 'W15-18', 'Fairness, MLOps, LLM/Embeddings, Final Project'),
]
print(f'{"Phase":<20} {"Weeks":<10} {"Topics"}')
print('-' * 80)
for phase, weeks_range, topics in phases:
    print(f'{phase:<20} {weeks_range:<10} {topics}')

In [ ]:
# === Key Concepts Learned (Visual Summary) ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) ML vs DL comparison
ax = axes[0]
categories = ['Feature\nEngineering', 'Data\nNeeded', 'Interpretability', 'Compute\nCost', 'Typical\nUse']
ml_scores = [5, 2, 5, 2, 3]   # Scale 1-5
dl_scores = [2, 5, 2, 5, 5]
x_pos = np.arange(len(categories))
width = 0.35
ax.bar(x_pos - width/2, ml_scores, width, label='Traditional ML', color='#FF6B6B', alpha=0.8)
ax.bar(x_pos + width/2, dl_scores, width, label='Deep Learning', color='#4ECDC4', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(categories, fontsize=8)
ax.set_ylabel('Level (1-5)')
ax.set_title('ML vs DL Comparison\nML \u8207 DL \u6bd4\u8f03', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 6)

# (2) Evaluation metrics learned
ax = axes[1]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC', 'MSE/MAE', 'R-squared']
metric_types = ['Classification', 'Classification', 'Classification', 'Classification',
                'Classification', 'Regression', 'Regression']
metric_colors = ['#FF6B6B' if t == 'Classification' else '#4ECDC4' for t in metric_types]
ax.barh(range(len(metrics_names)), [1]*len(metrics_names), color=metric_colors,
        edgecolor='white', height=0.7)
for i, name in enumerate(metrics_names):
    ax.text(0.5, i, name, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Evaluation Metrics Covered\n\u8a55\u4f30\u6307\u6a19\u7e3d\u89bd', fontsize=12, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#FF6B6B', label='Classification'),
                   Patch(color='#4ECDC4', label='Regression')], fontsize=9)

# (3) Skills roadmap
ax = axes[2]
ax.axis('off')
skills_text = """
SKILLS ACQUIRED

Data Skills:
  - EDA & Visualization
  - Feature Engineering
  - Data Preprocessing

Modeling Skills:
  - Regression & Classification
  - Ensemble Methods
  - Neural Networks (MLP/CNN/RNN)
  - Transformer / Attention

Engineering Skills:
  - sklearn Pipeline
  - Hyperparameter Tuning
  - Model Evaluation
  - MLOps Basics

Application Skills:
  - Text (TF-IDF, Embeddings)
  - RAG & Prompt Engineering
  - Fairness & Ethics
"""
ax.text(0.05, 0.95, skills_text, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title('Skills Roadmap / \u6280\u80fd\u8def\u7dda\u5716', fontsize=12, fontweight='bold')

plt.suptitle('Course Review: What We Learned / \u8ab2\u7a0b\u56de\u9867', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercises / \u7df4\u7fd2\u984c

### Exercise 1: Apply the Full Pipeline to a Different Dataset
### \u7df4\u7fd2 1\uff1a\u5c07\u5b8c\u6574 Pipeline \u61c9\u7528\u5230\u4e0d\u540c\u8cc7\u6599\u96c6

Apply the complete pipeline (EDA -> Preprocessing -> Model Comparison -> Evaluation) to the Iris dataset.

In [ ]:
# === Exercise 1: Apply Pipeline to Iris Dataset ===
from sklearn.datasets import load_iris

# TODO: Step 1 - Load Iris dataset
# iris = load_iris()
# X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
# y_iris = pd.Series(iris.target)

# TODO: Step 2 - Perform EDA (distribution plots, correlation matrix)

# TODO: Step 3 - Split data and create preprocessing pipeline

# TODO: Step 4 - Train at least 3 models and compare

# TODO: Step 5 - Evaluate the best model (confusion matrix, classification report)

# TODO: Step 6 - Generate a summary report

print('Exercise 1: Apply the complete ML pipeline to the Iris dataset.')
print('Follow the same structure as Parts 1-9 above.')

### Exercise 2: Add Cross-Validation to the Comparison Framework
### \u7df4\u7fd2 2\uff1a\u5728\u6bd4\u8f03\u6846\u67b6\u4e2d\u52a0\u5165\u4ea4\u53c9\u9a57\u8b49

Enhance the model comparison:
1. Use `RepeatedStratifiedKFold` (5-fold, 3 repeats)
2. Compute confidence intervals for accuracy
3. Use a paired t-test to compare the top 2 models

In [ ]:
# === Exercise 2: Enhanced Cross-Validation ===
from sklearn.model_selection import RepeatedStratifiedKFold

# TODO: Step 1 - Create RepeatedStratifiedKFold
# rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# TODO: Step 2 - Evaluate multiple models with repeated CV
# Store all fold scores for each model

# TODO: Step 3 - Compute 95% confidence interval for each model
# CI = mean +/- 1.96 * (std / sqrt(n_folds * n_repeats))

# TODO: Step 4 - Paired t-test between top 2 models
# from scipy import stats
# t_stat, p_val = stats.ttest_rel(scores_model1, scores_model2)

# TODO: Step 5 - Visualize with error bars and significance

print('Exercise 2: Implement enhanced cross-validation comparison.')
print('Expected: Confidence intervals and statistical significance testing.')

### Exercise 3: Create a Presentation-Ready Results Figure
### \u7df4\u7fd2 3\uff1a\u5efa\u7acb\u5c55\u793a\u7528\u7d50\u679c\u5716

Create a single, publication-quality figure that:
1. Summarizes the key results in a 2x2 layout
2. Uses consistent colors and fonts
3. Includes proper labels, legends, and annotations
4. Could be used directly in a presentation slide

In [ ]:
# === Exercise 3: Presentation-Ready Figure ===

# TODO: Create a 2x2 figure with:
# (1) Model comparison bar chart with error bars
# (2) Confusion matrix with percentage annotations
# (3) ROC curves with AUC values
# (4) Feature importance (top 8)

# Design guidelines:
# - Use fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# - Consistent color palette
# - All text readable at presentation size (fontsize >= 10)
# - Include a main title with the project name
# - Add subplot labels (a), (b), (c), (d)

print('Exercise 3: Create a presentation-ready 2x2 results figure.')
print('Tip: Use plt.rcParams to set global font sizes for consistency.')

---
## Summary & Course Conclusion / \u6559\u5b78\u91cd\u9ede\u6574\u7406\u8207\u8ab2\u7a0b\u7e3d\u7d50

### Key Takeaways / \u95dc\u9375\u8981\u9ede

| \u4e3b\u984c Topic | \u91cd\u9ede Key Point |
|------|------|
| End-to-End Pipeline | \u5b8c\u6574\u7684 ML \u5c08\u6848\u5305\u542b\u554f\u984c\u5b9a\u7fa9\u3001EDA\u3001\u524d\u8655\u7406\u3001\u8a13\u7df4\u3001\u8a55\u4f30\u3001\u5831\u544a |
| EDA | \u81ea\u52d5\u5316 EDA \u53ef\u5feb\u901f\u7406\u89e3\u8cc7\u6599\u7684\u5206\u5e03\u3001\u76f8\u95dc\u6027\u3001\u7f3a\u5931\u503c |
| Preprocessing | \u4f7f\u7528 Pipeline \u907f\u514d\u8cc7\u6599\u6d29\u6f0f\uff0c\u78ba\u4fdd\u53ef\u91cd\u73fe\u6027 |
| Model Comparison | \u8a13\u7df4\u591a\u500b\u6a21\u578b\uff0c\u7528\u4ea4\u53c9\u9a57\u8b49\u9032\u884c\u516c\u5e73\u6bd4\u8f03 |
| Tuning | GridSearchCV + \u9a57\u8b49\u66f2\u7dda\u5354\u52a9\u9078\u64c7\u6700\u4f73\u53c3\u6578 |
| Evaluation | \u6df7\u6dc6\u77e9\u9663\u3001ROC\u3001\u7279\u5fb5\u91cd\u8981\u6027\u63d0\u4f9b\u5168\u9762\u8a55\u4f30 |
| Interpretation | \u63cf\u8ff0\u4f9d\u8cf4\u5716\u3001\u7f6e\u63db\u91cd\u8981\u6027\u5e6b\u52a9\u7406\u89e3\u6a21\u578b\u6c7a\u7b56 |
| Report | \u7d50\u69cb\u5316\u5831\u544a\u8b93\u7d50\u679c\u53ef\u6e9d\u901a\u3001\u53ef\u91cd\u73fe |
| 18 Weeks | \u5f9e ML \u57fa\u790e\u5230 DL \u9032\u968e\u518d\u5230\u61c9\u7528\u5be6\u8e10 |

### \u8ab2\u7a0b\u7e3d\u7d50 Course Conclusion

\u5728\u9019 18 \u9031\u7684\u8ab2\u7a0b\u4e2d\uff0c\u6211\u5011\u5f9e ML \u7684\u57fa\u672c\u6982\u5ff5\u51fa\u767c\uff0c\u7d93\u904e\u5404\u7a2e\u7d93\u5178\u6f14\u7b97\u6cd5\u7684\u5b78\u7fd2\uff0c\u6df1\u5165\u6df1\u5ea6\u5b78\u7fd2\u67b6\u69cb\uff0c\u6700\u5f8c\u5230 MLOps \u8207 LLM \u61c9\u7528\u3002

Over 18 weeks, we progressed from basic ML concepts through classical algorithms, deep learning architectures, and finally to MLOps and LLM applications.

**\u672a\u4f86\u5b78\u7fd2\u5efa\u8b70 Future Learning:**
- \u53c3\u52a0 Kaggle \u7af6\u8cfd\u7df4\u529f
- \u95b1\u8b80\u8ad6\u6587\uff0c\u8ffd\u8e64\u6700\u65b0\u7814\u7a76
- \u5efa\u7acb\u500b\u4eba\u4f5c\u54c1\u96c6 (Portfolio)
- \u63a2\u7d22\u5c08\u7cbe\u9818\u57df (NLP, CV, MLOps, etc.)

---

**Congratulations on completing the ML/DL Visualization Course!**

**\u606d\u559c\u5b8c\u6210\u6a5f\u5668\u5b78\u7fd2/\u6df1\u5ea6\u5b78\u7fd2\u8996\u89ba\u5316\u8ab2\u7a0b\uff01**